In [4]:
# Embed all song chunks

import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import os
from tqdm import tqdm

load_dotenv()

from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

file_path = "/content/drive/MyDrive/GenAI Final Project Music&Memory/knowledge_base.csv"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"The file '{file_path}' was not found. Please ensure the file is uploaded to Google Drive and the path is correct.")

df = pd.read_csv(file_path)
chunks = df["text_chunk"].tolist()

def get_embeddings(texts, model="text-embedding-3-small", batch_size=100):
    """Embed a list of texts in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(input=batch, model=model)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
    return np.array(all_embeddings, dtype="float32")

# Get embedding shape
embeddings = get_embeddings(chunks)
print(f"Embedding shape: {embeddings.shape}")

# Save embeddings so that recomputation is not required
np.save("/content/drive/MyDrive/GenAI Final Project Music&Memory/embeddings.npy", embeddings)

Mounted at /content/drive


100%|██████████| 24/24 [00:21<00:00,  1.14it/s]


Embedding shape: (2381, 1536)


In [10]:
# Install FAISS

!pip install faiss-cpu

In [9]:
# Build the FAISS index

import faiss

# Load embeddings if needed
embeddings = np.load("/content/drive/MyDrive/GenAI Final Project Music&Memory/embeddings.npy")

# Build the index
dimension = embeddings.shape[1]  # 1536 for text-embedding-3-small
index = faiss.IndexFlatIP(dimension)  # Inner product (cosine similarity for normalized vectors)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

# Save the index
faiss.write_index(index, "/content/drive/MyDrive/GenAI Final Project Music&Memory/songs.index")

FAISS index built with 2381 vectors


In [12]:
# Save this as src/retrieval.py

import faiss
import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI()

def load_retrieval_system(index_path, kb_path):
    """Load the FAISS index and knowledge base."""
    index = faiss.read_index(index_path)
    df = pd.read_csv(kb_path)
    return index, df

def retrieve(query, index, df, k=20):
    """Retrieve top-k songs for a query string."""
    # Embed the query
    response = client.embeddings.create(
        input=[query], model="text-embedding-3-small"
    )
    query_vec = np.array([response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vec)

    # Search
    scores, indices = index.search(query_vec, k)

    # Return results
    results = df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results

In [13]:
# Test the retrieval
index, df = load_retrieval_system("/content/drive/MyDrive/GenAI Final Project Music&Memory/songs.index", "/content/drive/MyDrive/GenAI Final Project Music&Memory/knowledge_base.csv")
results = retrieve("popular soul and Motown songs in Detroit in the 1960s", index, df, k=10)
print(results[["song", "artist", "year", "similarity_score"]])

                          song  \
756                   Soul Man   
921      Dancing In The Street   
777                Shop Around   
1488                  Cruisin'   
810         Little Bit O' Soul   
1254             Soulful Strut   
132          Poor Side Of Town   
368                    My Girl   
492      If You Wanna Be Happy   
719   Papa Was A Rollin' Stone   

                                               artist  year  similarity_score  
756                                        Sam & Dave  1967          0.442117  
921                            Martha & The Vandellas  1964          0.433801  
777   The Miracles (featuring Bill "Smokey" Robinson)  1961          0.421136  
1488                                  Smokey Robinson  1980          0.409984  
810                               The Music Explosion  1967          0.404584  
1254                             Young-Holt Unlimited  1969          0.403116  
132                                     Johnny Rivers  1966      